# 08 · The whole setup from Python, comparing runs, and legacy inis

`posydon-grid explain` and `setup` are thin wrappers around three functions in `posydon/grids/setup/steps.py`, which you can call from a script or notebook:

| Function | Steps | Writes |
|---|---|---|
| `prepare(ini, ...)` | 1–3: read and check the ini, read the points, resolve the scenario, fill in defaults, pick extras and column lists | nothing |
| `explain_grid(ini, ...)` | `prepare`, then where every value comes from | nothing |
| `setup_grid(ini, ...)` | `prepare`, then 4–8 | the run directory |

This notebook uses them to compare two setups (release vs rerun), then reads an old-style (legacy) ini the way `posydon-setup-grid` users still will.

In [ ]:
from nb_env import env, fresh, write, show, short, grid_csv, tree, cli
ENV = env()

import sys
from pathlib import Path
import pandas as pd
from posydon.grids.setup.steps import explain_grid, prepare, setup_grid
from posydon.grids.setup.inlist import read_inlist

ZAMS = ENV.inlists / "r11701" / "ZAMS_models" / "zams_z1.42m2_y0.2703.data"

def grid_ini(name, grid_type="HMS-HMS", scenario_extra="", grid_extra=f"zams = {ZAMS}", rest=""):
    folder = fresh(name)
    grid_csv(folder)
    return write(folder / "grid.ini", f"""
        [grid]
        type = {grid_type}
        points = grid.csv
        run_directory = run
        {grid_extra}
        [scenario]
        release = v2.0
        inlist_repo = {ENV.inlists}
        {scenario_extra}
        [submission]
        backend = shell
        {rest}
        """)

## `prepare`: everything checked, nothing written

In [2]:
ini = grid_ini("08_release")
ready = prepare(ini)
print("run directory:", ready.run_directory)
print("grid type:    ", ready.config.grid.type, "/", ready.config.grid.mode)
print("inlist commit:", ready.resolved.commit)
print("parameters:   ", list(ready.parameters))
print("extras:       ", list(ready.extras))
print("column lists: ", {k: short(v, ENV.release) for k, v in ready.columns.items()})
ready.grid_df

run directory: /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/08_release/run
grid type:     HMS-HMS / fixed
inlist commit: 115de83c33bad5013cb09837fcc679d1d45e816b
parameters:    ['initial_z', 'Zbase', 'm1', 'm2', 'initial_period_in_days']
extras:        ['makefile_binary', 'makefile_star', 'mesa_star1_extras', 'mesa_star2_extras', 'binary_run', 'star_run', 'posydon_binary_extras', 'posydon_star_binary_extras']
column lists:  {'history_columns': 'common/history_columns.list', 'binary_history_columns': 'common/binary_history_columns.list', 'profile_columns': 'common/profile_columns.list'}


,initial_z,Zbase,m1,m2,initial_period_in_days
0,0.0142,0.0142,30.0,20.0,10.0
1,0.0142,0.0142,50.0,25.0,3.0
2,0.0142,0.0142,12.5,10.0,100.0


## `explain_grid` and `setup_grid`

In [3]:
text = explain_grid(ini)
print(len(text.splitlines()), "lines;", sum(1 for l in text.splitlines() if l.startswith("  ") and not l.startswith("    ")), "keys not from MESA's defaults")

709 lines; 357 keys not from MESA's defaults


In [4]:
run_release = setup_grid(ini)
tree(run_release)

binary/inlist1
binary/inlist2
binary/inlist_project
binary/make/  (1 files)
binary/src/  (3 files)
column_lists/binary_history_columns.list
column_lists/history_columns.list
column_lists/profile_columns.list
grid_command.sh
mk
setup_record.ini
star1/make/  (1 files)
star1/src/  (2 files)
star2/make/  (1 files)
star2/src/  (2 files)


## Compare two setups: release v2.0 vs a rerun

Set up the same grid with a rerun and compare the two run directories: every inlist key by key, and every other file by content. This is how you check what a change to the inlists or to the setup code does to a real run directory.

In [5]:
def compare_runs(a, b):
    """Differences between two run directories: inlist keys, then other files."""
    a, b = Path(a), Path(b)
    files = sorted({p.relative_to(a) for p in a.rglob("*") if p.is_file()} |
                   {p.relative_to(b) for p in b.rglob("*") if p.is_file()})
    rows, other = [], []
    for rel in files:
        fa, fb = a / rel, b / rel
        if not (fa.is_file() and fb.is_file()):
            other.append(f"{rel}: only in {'first' if fa.is_file() else 'second'}")
            continue
        if rel.parts[0] in ("binary", "star1", "star2") and rel.name.startswith("inlist"):
            sa, sb = read_inlist(fa), read_inlist(fb)
            for section in sorted(set(sa) | set(sb)):
                va, vb = sa.get(section, {}), sb.get(section, {})
                for key in list(dict.fromkeys(list(va) + list(vb))):
                    if va.get(key) != vb.get(key):
                        rows.append((str(rel), section, key, va.get(key), vb.get(key)))
        elif fa.read_bytes() != fb.read_bytes() and rel.name != "setup_record.ini":
            other.append(f"{rel}: content differs")
    return pd.DataFrame(rows, columns=["file", "section", "key", "first", "second"]), other

ini_rerun = grid_ini("08_rerun", scenario_extra="rerun = matthias_lbv_thermo_dedt_noAgeLimit")
run_rerun = setup_grid(ini_rerun)
keys, other = compare_runs(run_release, run_rerun)
print("\n".join(other))
keys

binary/src/run_binary_extras.f: content differs
binary/src/run_star_extras.f: content differs
grid_command.sh: content differs
mk: content differs
star1/src/run_star_extras.f: content differs
star2/src/run_star_extras.f: content differs


,file,section,key,first,second
0,binary/inlist1,controls,max_age,1.4e10,-1
1,binary/inlist1,controls,thermohaline_coeff,0.0d0,17.5d0
2,binary/inlist1,controls,surf_avg_tau_min,0.0,10.0
3,binary/inlist1,controls,surf_avg_tau,10.0,100.0
4,binary/inlist1,controls,surf_w_div_w_crit_limit,0.99d0,0.95d0
5,binary/inlist1,controls,surf_w_div_w_crit_tol,0.05d0,0.1d0
6,binary/inlist1,controls,max_logT_for_k_below_const_q,5,1.0d2
7,binary/inlist1,controls,max_q_for_k_below_const_q,1.0d0,0.995d0
8,binary/inlist1,controls,min_q_for_k_below_const_q,0.999d0,0.995d0
9,binary/inlist1,controls,max_logT_for_k_const_mass,6,1.0d2


Paths that contain the run directory (`inlist_names`, the scripts) differ only because the two runs live in different folders. Everything else is the rerun's physics.

## Legacy ini files

Legacy inis (`[slurm]`, `[mesa_inlists]`, `[mesa_extras]`, `[run_parameters]`) still work. `legacy.read_legacy_config` reads them into the same `GridConfig`. Three things are not in a legacy file and come from the command line: `--grid-type` (fixed or dynamic), `--submission-type` (shell or slurm) and `--run-directory`.

Here is a legacy HMS-HMS ini that spells out the v2.0 files itself, without a scenario:

In [6]:
folder = fresh("08_legacy")
grid_csv(folder)
defaults = ENV.inlists / "r11701" / "mesa" / "default" / "defaults"
common = ENV.release / "common"
legacy_ini = write(folder / "legacy.ini", f"""
    [slurm]
    number_of_nodes = 1
    number_of_mpi_tasks = 1
    number_of_cpus_per_task = 1
    job_array = True
    partition = public-cpu
    account = my_account
    walltime = '2-00:00:00'
    email = me@example.org

    [mesa_inlists]
    zams_filename = {ZAMS}
    single_star_grid = False
    binary_controls_mesa_defaults = {defaults}/binary/binary_controls.defaults
    binary_job_mesa_defaults = {defaults}/binary/binary_job.defaults
    star1_job_mesa_defaults = {defaults}/star/star_job.defaults
    star1_controls_mesa_defaults = {defaults}/star/controls.defaults
    star2_job_mesa_defaults = {defaults}/star/star_job.defaults
    star2_controls_mesa_defaults = {defaults}/star/controls.defaults
    star1_controls_posydon_defaults = {common}/binary/inlist1
    star1_job_posydon_defaults = {common}/binary/inlist1
    star2_controls_posydon_defaults = {common}/binary/inlist2
    star2_job_posydon_defaults = {common}/binary/inlist2
    binary_controls_posydon_defaults = {common}/binary/inlist_project
    binary_job_posydon_defaults = {common}/binary/inlist_project
    star_history_columns = {common}/history_columns.list
    binary_history_columns = {common}/binary_history_columns.list
    profile_columns = {common}/profile_columns.list
    history_interval = 1
    binary_history = True
    history_star1 = True
    final_profile_star1 = False
    final_model_star1 = True
    history_star2 = True
    final_profile_star2 = False
    final_model_star2 = True

    [mesa_extras]
    makefile_binary = ${{MESA_DIR}}/binary/work/make/makefile
    makefile_star = ${{MESA_DIR}}/star/work/make/makefile
    mesa_binary_extras = ${{MESA_DIR}}/binary/work/src/run_binary_extras.f
    mesa_star_binary_extras = ${{MESA_DIR}}/binary/work/src/run_star_extras.f
    mesa_star1_extras = {common}/binary/src/run_star_extras.f
    mesa_star2_extras = {common}/binary/src/run_star_extras.f
    binary_run = ${{MESA_DIR}}/binary/work/src/binary_run.f
    star_run = ${{MESA_DIR}}/star/work/src/run.f
    posydon_binary_extras = {common}/binary/src/run_binary_extras.f
    posydon_star_binary_extras = {common}/binary/src/run_star_extras.f

    [run_parameters]
    grid = {folder}/grid.csv
    """)

from posydon.grids.setup.legacy import read_legacy_config
legacy = read_legacy_config(legacy_ini, grid_type="fixed", submission_type="shell",
                            run_directory=folder / "run")
print("format:", legacy.source_format, "  grid type:", legacy.grid.type)
print("notes: ", legacy.notes)
for layer, files in legacy.inlists.items():
    print(layer, {t: [short(p, ENV.inlists / "r11701") for p in ps] for t, ps in files.targets.items()})

format: legacy   grid type: None
notes:  ['legacy ini format; see the grid documentation for the new [grid]/[scenario]/[user_inlists] format', 'no scenario: the grid type is not stated in the file']
mesa_inlists {'binary_controls': ['mesa/default/defaults/binary/binary_controls.defaults'], 'binary_job': ['mesa/default/defaults/binary/binary_job.defaults'], 'star1_job': ['mesa/default/defaults/star/star_job.defaults'], 'star1_controls': ['mesa/default/defaults/star/controls.defaults'], 'star2_job': ['mesa/default/defaults/star/star_job.defaults'], 'star2_controls': ['mesa/default/defaults/star/controls.defaults']}
posydon_inlists {'star1_controls': ['releases/v2.0/common/binary/inlist1'], 'star1_job': ['releases/v2.0/common/binary/inlist1'], 'star2_controls': ['releases/v2.0/common/binary/inlist2'], 'star2_job': ['releases/v2.0/common/binary/inlist2'], 'binary_controls': ['releases/v2.0/common/binary/inlist_project'], 'binary_job': ['releases/v2.0/common/binary/inlist_project']}
user_in

The keys are sorted into layers by their suffix: `*_mesa_defaults` → `mesa_inlists`; `*_posydon_defaults` and `*_special` → `posydon_inlists`; `*_user` → `user_inlists`. Without a `scenario` line a legacy ini does not say its grid type (`None`), and the setup works the old way, from the files it lists.

### Legacy and new format give the same run directory

Set up the legacy ini with `posydon-grid setup --grid-type ...` and compare with the new-format `08_release` run from above:

In [7]:
cli("setup", legacy_ini, "--grid-type", "fixed", "--run-directory", folder / "run", "--submission-type", "shell")
keys, other = compare_runs(run_release, folder / "run")
print("\n".join(other))
keys

--- stderr ---
/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/posydon/grids/setup/steps.py:159: InappropriateValueWarning: 'no scenario: the grid type is not stated in the file'
/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/posydon/grids/setup/steps.py:208: ReplaceValueWarning: 'Section mesa_extras value mesa_binary_extras is being set to None'
grid_command.sh: content differs
mk: content differs


,file,section,key,first,second
0,binary/inlist_project,binary_job,inlist_names(1),'/Users/max/Documents/POSYDON/.claude/worktree...,'/Users/max/Documents/POSYDON/.claude/worktree...
1,binary/inlist_project,binary_job,inlist_names(2),'/Users/max/Documents/POSYDON/.claude/worktree...,'/Users/max/Documents/POSYDON/.claude/worktree...


The only differences are the paths that name the run directory itself.

### The deprecated `posydon-setup-grid`

It still works with the same options and output, and tells you the new command:

In [8]:
import subprocess
result = subprocess.run([sys.executable, str(ENV.root / "bin" / "posydon-setup-grid"),
                         "--inifile", str(legacy_ini), "--grid-type", "fixed",
                         "--run-directory", str(fresh("08_legacy_old")), "--submission-type", "shell"],
                        capture_output=True, text=True)
print([line for line in result.stderr.splitlines() if "deprecated" in line][0].split("DeprecationWarning: ")[-1])

"posydon-setup-grid is deprecated and will be removed; use 'posydon-grid setup /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/08_legacy/legacy.ini --grid-type fixed --run-directory /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/08_legacy_old --submission-type shell' instead (posydon-grid has the commands types, init, explain, setup and run)"


### A legacy ini with a scenario

Most v2 production inis instead name a scenario: `scenario = ['posydon', '<branch>-<commit>', '<system type>']`. That checks out the named commit of the inlist repository (by default in `~/.posydon_mesa_inlists`) and finds the files by the old rules (`scenario.scenario_files`), so old grids keep their physics. `MS-MS` is read as HMS-HMS with a note (G4), and `grid_params/` paths, which POSYDON no longer ships, are read from the inlist repository at a pinned commit (`support.py`). This notebook does not run one, since it would check out an old commit in your clone; the test suite replays all 797 v2 inis on yggdrasil this way (`grid_rewrite_tests/`).

## Try it

- Change one value in `common/binary/inlist1` of a copy of the inlist repository, point `inlist_repo` at the copy, and use `compare_runs` to see exactly what changes in the run directory.
- Compare `HeMS-HMS` with and without `zams = ...` in `[grid]`: star 1 is formed either way (S2).